# 01 — Synthetic Data Understanding

> ⚠️ **SYNTHETIC DATA — NOT REAL SURVEY RESPONSES**
>
> This notebook analyzes a **synthetic** dataset (`data/synthetic/`)
> generated by a Large Language Model to test whether the machine learning
> pipeline scales from 39 real responses to a larger sample (n=1000).
>
> - **Source:** Large Language Model (LLM)
> - **Purpose:** Pipeline scaling validation, model comparison
> - **NOT used for:** Claims about real students, academic stress patterns,
>   or generalizable findings
>
> All real-data findings appear in the primary `notebooks/` directory.

---

**Input:**  `data/synthetic/student_stress_synthetic_1000.csv`  (1,000 rows × 35 cols)

**What this notebook does:**
1. Loads the synthetic dataset
2. Inspects shape, columns, data types
3. Checks for missing values
4. Examines the key survey-variable distributions
5. Compares structure against the real dataset
6. Establishes the foundation for the synthetic pipeline

In [1]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Setup complete.")

Setup complete.


## 1. Load the Synthetic Dataset

The synthetic CSV was generated by an LLM to structurally mirror the real
survey dataset. It uses the same 34 column names (raw, uncleaned form) as the
real `student_stress.csv`, plus one additional column (`Data_Source`) that
self-identifies the file as synthetic.

In [3]:
# Load the synthetic dataset
df = pd.read_csv("../data/synthetic/student_stress_synthetic_1000.csv")

print("Shape:", df.shape)
print()
print("First 3 columns:")
print(df.columns[:3].tolist())
print()
print("Last 3 columns:")
print(df.columns[-3:].tolist())

Shape: (1000, 35)

First 3 columns:
['Timestamp', 'Research Participation Consent\nI have read the information provided above and voluntarily agree to participate in this research study. I understand that my responses will be anonymous and used only for academic and research purposes.', 'How old are you? ']

Last 3 columns:
['How often do concerns about your future career or employment cause you stress?', 'How satisfied are you with your current academic performance?', 'Data_Source']


## 2. Verify Data Source

The file includes a `Data_Source` column. This should be `"Synthetic"` for
every row — confirming this data is not being confused with real responses.

In [4]:
# Confirm every row is marked as Synthetic
print("Data_Source unique values:")
print(df["Data_Source"].value_counts())
print()

assert (df["Data_Source"] == "Synthetic").all(), "ERROR: file is not fully synthetic!"
print("✅ Confirmed: all 1000 rows are marked Synthetic")

Data_Source unique values:
Data_Source
Synthetic    1000
Name: count, dtype: int64

✅ Confirmed: all 1000 rows are marked Synthetic


## 3. Column Overview

The synthetic file uses the **same 34 columns** as the real raw data
(plus the `Data_Source` marker). This means the same cleaning and
feature-engineering logic can be reused without modification.

In [5]:
# List all columns with their dtype
print("All columns and dtypes:")
print()
for i, (col, dtype) in enumerate(zip(df.columns, df.dtypes), 1):
    # Truncate long column names for readability
    short = col[:70] + "..." if len(col) > 70 else col
    print(f"  {i:2d}. [{dtype}] {short}")

All columns and dtypes:

   1. [str] Timestamp
   2. [str] Research Participation Consent
I have read the information provided ab...
   3. [str] How old are you? 
   4. [str] What is your gender? 
   5. [str]   Which university are you currently studying in?  
   6. [str]    What program are you currently enrolled in?  
   7. [str]   Which year are you currently studying in?  
   8. [str]   Current GPA/CGPA  
   9. [str] On average, how many hours do you study per day (outside class)?
  10. [str] What is your approximate class attendance?
  11. [int64] How often do you feel overwhelmed by assignments, projects, and deadli...
  12. [str] How many major exams, tests, or assessments have you completed during ...
  13. [str] How many hours of sleep do you get per night? 
  14. [str] On average, how much time do you spend on social media each day?
  15. [str] How often do you engage in physical activity or exercise?
  16. [str]   Part-time Job  
  17. [str] On average, what is your total da

## 4. Missing Values

If the LLM generated the data properly, there should be **no missing
values**. A high missing count would mean the file is unusable.

In [6]:
missing = df.isnull().sum()
n_missing = missing.sum()

print(f"Total missing values: {n_missing}")
print()

if n_missing == 0:
    print("✅ No missing values — dataset is complete")
else:
    print("⚠️ Missing values detected in:")
    print(missing[missing > 0].sort_values(ascending=False))

Total missing values: 0

✅ No missing values — dataset is complete


## 5. Target Variable — Stress Questions

Unlike the real data, the synthetic file does **not** include a pre-computed
`stress_score` or `stress_level`. We will construct these later (in notebook 02)
using the same reverse-scoring and tertile-split approach as the real pipeline.

For now, let's check the 10 stress questions.

In [7]:
# The last ~14 columns include the 10 stress questions
# Identify them by their distinctive text
stress_q_cols = [c for c in df.columns if "During the last month" in c]
print(f"Found {len(stress_q_cols)} stress-question columns:")
for i, c in enumerate(stress_q_cols, 1):
    print(f"  {i:2d}. {c[:75]}...")

Found 10 stress-question columns:
   1. During the last month, how often have you felt nervous or stressed because ...
   2. During the last month, how often have you felt that your academic responsib...
   3. During the last month, how often have you worried about your academic perfo...
   4. During the last month, how often have you felt confident in your ability to...
   5. During the last month, how often have you felt that things were going well ...
   6. During the last month, how often have you felt overwhelmed by the amount of...
   7. During the last month, how often have you been able to stay calm when facin...
   8. During the last month, how often have you felt in control of your studies a...
   9. During the last month, how often have you felt frustrated because of situat...
  10. During the last month, how often have you felt that your academic problems ...


### Distribution of each stress question (1–5 scale)

A well-generated dataset should show a **spread across all 5 answer
options**. Bunching at a single value would mean the data is degenerate.

In [8]:
for i, col in enumerate(stress_q_cols, 1):
    print(f"\n--- Q{i} ---")
    print(df[col].value_counts().sort_index())


--- Q1 ---
During the last month, how often have you felt nervous or stressed because of your studies?
1    125
2    244
3    270
4    216
5    145
Name: count, dtype: int64

--- Q2 ---
During the last month, how often have you felt that your academic responsibilities were becoming difficult to manage?  
1    122
2    234
3    294
4    221
5    129
Name: count, dtype: int64

--- Q3 ---
During the last month, how often have you worried about your academic performance or grades?  
1    122
2    235
3    265
4    252
5    126
Name: count, dtype: int64

--- Q4 ---
During the last month, how often have you felt confident in your ability to handle academic challenges?  
1    110
2    247
3    309
4    222
5    112
Name: count, dtype: int64

--- Q5 ---
During the last month, how often have you felt that things were going well in your academic life?  
1    105
2    245
3    300
4    230
5    120
Name: count, dtype: int64

--- Q6 ---
During the last month, how often have you felt overwhelmed b

## Summary — Observations

- ✅ Dataset contains **1,000 synthetic rows** and **35 columns** (34 real-data columns + `Data_Source` marker)
- ✅ No missing values
- ✅ All rows are correctly marked `Data_Source = "Synthetic"`
- ✅ The 10 stress questions show spread across the 1–5 answer range
- ✅ The dataset structure mirrors the real raw data exactly

**Next notebook (`02_synth_cleaning_and_target.ipynb`):**
- Drop `Timestamp`, consent column, and `Data_Source`
- Rename columns to `snake_case` (matching the real pipeline)
- Reverse-score `stress_q4`, `stress_q5`, `stress_q7`, `stress_q8`
- Compute `stress_score`
- Create `stress_level` via tertile split
- Save cleaned synthetic data to `data/synthetic/processed/`

**Note:** This is synthetic data. Results from it are for pipeline validation
only and do not represent real students.